In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score
!pip install catboost
from catboost import CatBoostClassifier
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.5 MB/s eta 0:00:00


In [2]:
df1=pd.read_csv('merged_with_tweets_training.csv')
df2=pd.read_csv('predictions.csv')
df1["tweets"] = df1["tweets"].str.lower()
df2["tweets"] = df2["tweets"].str.lower()
df1["tweets"] = df1["tweets"].str.strip()
df2["tweets"] = df2["tweets"].str.strip()
df = df1.merge(df2, on=["tweets", "congestion_level", "timestamp"], how="inner")
df = df.drop(columns=["Unnamed: 0"])
df = df.drop_duplicates()
df

,timestamp,geometry,tweets,Boro,Vol,congestion,congestion_level,Temperature,Condition,Wind,Humidity,Pressure,Visibility,predicted_label,confidence,entropy
0,2025-02-22 03:30:00,POLYGON ((-73.98203507773665 40.65524805627688...,"['just drove to the store and it was a breeze,...",Brooklyn,8,-1.589893,0,-6 °C,Clear.,9 km/h,54%,1029 mbar,16 km,0,0.999868,0.001396
1,2024-11-09 09:15:00,POLYGON ((-73.93291959036472 40.82741450154307...,"[""just headed out for a run and it's a beautif...",Manhattan,264,0.560178,0,7 °C,Sunny.,No wind,40%,1027 mbar,16 km,0,0.999887,0.001210
2,2025-03-06 13:45:00,POLYGON ((-73.97178007473232 40.79434366943585...,"[""just braved the chilly morning commute, it's...",Manhattan,123,0.635957,0,11 °C,Overcast.,NaN,50%,992 mbar,16 km,0,0.999803,0.002003
3,2024-04-04 15:30:00,POLYGON ((-73.98780645780917 40.67354750428635...,"['woke up to a pretty chilly grey day, 9°c out...",Brooklyn,151,0.744760,0,9 °C,Overcast.,6 km/h,52%,996 mbar,16 km,0,0.999872,0.001364
4,2025-01-08 21:15:00,POLYGON ((-73.91101950432183 40.87001883438244...,"['just got out of my car, finally parked in a ...",Manhattan,6,-0.134900,0,-5 °C,Passing clouds.,NaN,46%,1010 mbar,16 km,0,0.999492,0.004536
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1450,2025-06-21 12:30:00,"POLYGON ((-73.9743296272132 40.78160532554132,...","[""i just saw someone blocking the bike lane on...",Manhattan,223,0.294327,0,28 °C,Sunny.,No wind,44%,1020 mbar,16 km,0,0.997050,0.020992
1451,2025-06-22 06:30:00,"POLYGON ((-73.9743296272132 40.78160532554132,...",['just got a parking ticket on columbus avenue...,Manhattan,72,-1.557482,0,24 °C,Sunny.,9 km/h,69%,1018 mbar,16 km,0,0.996148,0.026403
1452,2025-07-08 09:30:00,POLYGON ((-73.96628726515767 40.58021135286303...,"[""driving down ocean parkway and there's a car...",Brooklyn,27,0.974278,0,28 °C,Sunny.,NaN,70%,1015 mbar,16 km,0,0.959875,0.181338
1453,2025-07-08 09:30:00,POLYGON ((-73.96628726515767 40.58021135286303...,"[""just got a posted parking sign violation on ...",Brooklyn,27,0.974278,0,28 °C,Sunny.,NaN,70%,1015 mbar,16 km,0,0.998347,0.012785


In [3]:
from shapely import wkt
def get_centroid(polygon_str):
    geom = wkt.loads(polygon_str)
    return geom.centroid.x, geom.centroid.y

df[["lon", "lat"]] = df["geometry"].apply(
    lambda x: pd.Series(get_centroid(x))
)

df = df.drop(columns=["geometry"])
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp")
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]
train_df = train_df.drop("tweets", axis=1)
train_df = train_df.drop("Vol", axis=1)
train_df = train_df.drop("congestion", axis=1)
X_train = train_df.drop("congestion_level", axis=1)
y_train = train_df["congestion_level"]
test_df = test_df.drop("tweets", axis=1)
test_df = test_df.drop("Vol", axis=1)
test_df = test_df.drop("congestion", axis=1)
X_test = test_df.drop("congestion_level", axis=1)
y_test = test_df["congestion_level"]
X_train["Wind"] = X_train["Wind"].replace("No wind", np.nan)
X_test["Wind"] = X_test["Wind"].replace("No wind", np.nan)
def extract_number(column):
    return column.astype(str).str.extract(r'([-+]?\d*\.\d+|\d+)').astype(float)
X_train["Visibility"] = extract_number(X_train["Visibility"])
X_test["Visibility"] = extract_number(X_test["Visibility"])
X_train["Temperature"] = extract_number(X_train["Temperature"])
X_test["Temperature"] = extract_number(X_test["Temperature"])
X_train["Wind"] = X_train["Wind"].replace("No wind", "0")
X_test["Wind"] = X_test["Wind"].replace("No wind", "0")
X_train["Wind"] = extract_number(X_train["Wind"])
X_test["Wind"] = extract_number(X_test["Wind"])
X_train["Humidity"] = extract_number(X_train["Humidity"])
X_test["Humidity"] = extract_number(X_test["Humidity"])
X_train["Pressure"] = extract_number(X_train["Pressure"])
X_test["Pressure"] = extract_number(X_test["Pressure"])
X_train["month"] = X_train["timestamp"].dt.month
X_train["day"] = X_train["timestamp"].dt.day
X_train["weekday"] = X_train["timestamp"].dt.weekday
X_train["hour"] = X_train["timestamp"].dt.hour
X_train["minute"] = X_train["timestamp"].dt.minute
X_test["month"] = X_test["timestamp"].dt.month
X_test["day"] = X_test["timestamp"].dt.day
X_test["weekday"] = X_test["timestamp"].dt.weekday
X_test["hour"] = X_test["timestamp"].dt.hour
X_test["minute"] = X_test["timestamp"].dt.minute
X_train = X_train.drop(columns=["timestamp"])
X_test = X_test.drop(columns=["timestamp"])
cat_features = ['Boro', 'Condition']
for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

model = CatBoostClassifier()
model.fit(X_train, y_train,cat_features=cat_features)

Learning rate set to 0.079366
0:	learn: 0.9677627	total: 69.1ms	remaining: 1m 8s
1:	learn: 0.8663644	total: 86.7ms	remaining: 43.3s
2:	learn: 0.7802988	total: 104ms	remaining: 34.6s
3:	learn: 0.7073698	total: 129ms	remaining: 32s
4:	learn: 0.6505237	total: 142ms	remaining: 28.3s
5:	learn: 0.5968814	total: 152ms	remaining: 25.2s
6:	learn: 0.5517706	total: 164ms	remaining: 23.3s
7:	learn: 0.5089361	total: 178ms	remaining: 22.1s
8:	learn: 0.4667491	total: 189ms	remaining: 20.9s
9:	learn: 0.4328137	total: 208ms	remaining: 20.6s
10:	learn: 0.4041968	total: 229ms	remaining: 20.6s
11:	learn: 0.3743781	total: 243ms	remaining: 20s
12:	learn: 0.3504452	total: 255ms	remaining: 19.4s
13:	learn: 0.3281250	total: 262ms	remaining: 18.4s
14:	learn: 0.3070283	total: 267ms	remaining: 17.5s
15:	learn: 0.2873978	total: 283ms	remaining: 17.4s
16:	learn: 0.2684249	total: 298ms	remaining: 17.3s
17:	learn: 0.2510178	total: 311ms	remaining: 17s
18:	learn: 0.2379315	total: 327ms	remaining: 16.9s
19:	learn: 0.22

CatBoostClassifier()

In [4]:
from sklearn.metrics import f1_score, accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
y_pred = model.predict(X_test)
print(f1_score(y_test, y_pred, average='micro'))
print(precision_score(y_test, y_pred, average="macro"))
print(recall_score(y_test, y_pred, average="macro"))
print(classification_report(y_test,y_pred))

0.8345864661654135
0.829908103592314
0.8179392126602284
              precision    recall  f1-score   support

           0       0.84      0.91      0.88       123
           1       0.76      0.66      0.71        73
           2       0.89      0.89      0.89        70

    accuracy                           0.83       266
   macro avg       0.83      0.82      0.82       266
weighted avg       0.83      0.83      0.83       266



In [12]:
def objective(trial):
    """Define Optuna objective for CatBoostClassifier"""

    params =  {
        'depth': trial.suggest_int('depth', 4, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'iterations': trial.suggest_int('iterations', 500, 3000),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 20.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 5.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0),
        'rsm': trial.suggest_float('rsm', 0.5, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
        'verbose': 0,
        'eval_metric': 'MultiClass',
        'use_best_model': True
    }

    optuna_model = CatBoostClassifier(**params)
    optuna_model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_test, y_test),
        early_stopping_rounds=50
    )

    y_pred = optuna_model.predict(X_test)
    score = recall_score(y_test, y_pred, average='macro')
    return score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print("Best trial:")
trial = study.best_trial


[I 2026-03-21 23:14:03,049] A new study created in memory with name: no-name-2e66b014-99d7-4991-bbde-8dfda1bce350
[I 2026-03-21 23:14:04,195] Trial 0 finished with value: 0.9052845528455284 and parameters: {'depth': 5, 'learning_rate': 0.012633556587408565, 'iterations': 1134, 'l2_leaf_reg': 4.659056980901298, 'border_count': 252, 'bagging_temperature': 4.418911532828702, 'random_strength': 9.003079866065175, 'rsm': 0.9148104313206964, 'min_data_in_leaf': 36, 'grow_policy': 'Depthwise'}. Best is trial 0 with value: 0.9052845528455284.
[I 2026-03-21 23:14:12,527] Trial 1 finished with value: 0.917479674796748 and parameters: {'depth': 12, 'learning_rate': 0.09260520224636194, 'iterations': 1226, 'l2_leaf_reg': 1.449453723860342, 'border_count': 213, 'bagging_temperature': 3.956985238353276, 'random_strength': 8.099185823910764, 'rsm': 0.6345932856424952, 'min_data_in_leaf': 10, 'grow_policy': 'SymmetricTree'}. Best is trial 1 with value: 0.917479674796748.
[I 2026-03-21 23:14:15,767] Tr

Best trial:


In [13]:
print('Number of finished trials: {}'.format(len(study.trials)))
print('Best trial:')
trial = study.best_trial

print('  Value: {}'.format(trial.value))
print('  Params: ')

for key, value in trial.params.items():
    print('    {}: {}'.format(key, value))

Number of finished trials: 50
Best trial:
  Value: 0.917479674796748
  Params: 
    depth: 12
    learning_rate: 0.09260520224636194
    iterations: 1226
    l2_leaf_reg: 1.449453723860342
    border_count: 213
    bagging_temperature: 3.956985238353276
    random_strength: 8.099185823910764
    rsm: 0.6345932856424952
    min_data_in_leaf: 10
    grow_policy: SymmetricTree


In [14]:
params = trial.params
model = CatBoostClassifier(**params)
model.fit(X_train, y_train,cat_features=cat_features)
y_pred = model.predict(X_test)
print(f1_score(y_test, y_pred, average='micro'))
print(precision_score(y_test, y_pred, average="macro"))
print(recall_score(y_test, y_pred, average="macro"))
print(classification_report(y_test,y_pred))

0:	learn: 1.0073832	total: 225ms	remaining: 4m 35s
1:	learn: 0.9171857	total: 366ms	remaining: 3m 43s
2:	learn: 0.8172709	total: 370ms	remaining: 2m 30s
3:	learn: 0.7319844	total: 436ms	remaining: 2m 13s
4:	learn: 0.6521205	total: 440ms	remaining: 1m 47s
5:	learn: 0.5937746	total: 457ms	remaining: 1m 32s
6:	learn: 0.5525896	total: 459ms	remaining: 1m 19s
7:	learn: 0.4996473	total: 462ms	remaining: 1m 10s
8:	learn: 0.4641473	total: 533ms	remaining: 1m 12s
9:	learn: 0.4315502	total: 600ms	remaining: 1m 12s
10:	learn: 0.3973313	total: 617ms	remaining: 1m 8s
11:	learn: 0.3714914	total: 627ms	remaining: 1m 3s
12:	learn: 0.3441881	total: 640ms	remaining: 59.7s
13:	learn: 0.3165455	total: 660ms	remaining: 57.1s
14:	learn: 0.2962495	total: 769ms	remaining: 1m 2s
15:	learn: 0.2861247	total: 897ms	remaining: 1m 7s
16:	learn: 0.2710740	total: 916ms	remaining: 1m 5s
17:	learn: 0.2557159	total: 919ms	remaining: 1m 1s
18:	learn: 0.2468251	total: 949ms	remaining: 1m
19:	learn: 0.2365067	total: 1.06s	